In [ ]:
#importing the nessary libraries
import pandas as pd                    
import numpy as np                     
import matplotlib.pyplot as plt       
import matplotlib                     
import shap                             
import joblib                          
import warnings
warnings.filterwarnings("ignore")      
 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier       
from sklearn.cluster import KMeans
 

shap.initjs()

In [ ]:
#Loading data from the dataset
df = pd.read_csv("../data/diabetes_data.csv")

#Defining the features and target variable
#These are the features we will use for our model. We will also define the target variable, which is the diabetes stage.
FEATURES = ["Age", "gender", "ethnicity", "education_level", "income_level",
    "employment_status", "smoking_status", "alcohol_consumption_per_week",
    "physical_activity_minutes_per_week", "diet_score", "sleep_hours_per_day",
    "screen_time_hours_per_day", "family_history_diabetes", "hypertension_history",
    "cardiovascular_history", "bmi", "waist_to_hip_ratio", "systolic_bp",
    "diastolic_bp", "heart_rate", "cholesterol_total", "hdl_cholesterol",
    "ldl_cholesterol", "triglycerides", "glucose_fasting", "glucose_postprandial",
    "insulin_level", "hba1c", "diabetes_risk_score"]

TARGET = "diabetes_stage"

#we are encoding the categorical features using label encoding. This is because SHAP can only handle numerical features.
categorical_cols = [ "gender", "ethnicity", "education_level", "income_level",
    "employment_status", "smoking_status"]

#storing each encoder so we can inverse transform the values later if needed
encoders = {}
for col in categorical_cols:
    le = LabelEncoder()                     # Create a LabelEncoder instance
    df[col] = le.fit_transform(df[col])     # Replace text with numbers
    encoders[col] = le                      # Save the encoder

X = df[FEATURES]                             # Feature matrix (all input columns)
y = df["diabetes_stage_encoded"]             # Encoded target labels
 
# Split into training and test sets (80% train, 20% test)
# random_state=42 makes the split reproducible — you get the same split every run
X_train, X_test, y_train, y_test = train_test_split(  X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
#Training the XGBoost model on the training data.
clf_model = XGBClassifier(
    n_estimators=200,       
    max_depth=6,            
    learning_rate=0.1,      
    use_label_encoder=False,
    eval_metric="mlogloss", 
    random_state=42        
)
 
clf_model.fit(X_train, y_train)   # Train on the training data


In [ ]:
#SHAP for risk classification

# The TreeExplainer is designed for tree-based models like XGBoost, Random Forest, Decision Trees.
# Uses the model's internal tree structure to compute SHAP values efficiently.
clf_explainer = shap.TreeExplainer(clf_model)

# we use the test set 
X_test_sample = X_test.sample(n = 1000, random_state=42)  

# shap_values_clf is a 3D array
# Each entry shap_values_clf[i][j][k] represents the SHAP value for the j-th feature of the i-th sample for class k.
shap_values_clf = clf_explainer.shap_values(X_test_sample)

print(f"SHAP values shape: {np.array(shap_values_clf).shape}")  


# The beeswarm plot shows both imporance and direction
# x-axis: SHAP value (positive = increases risk, negative = decreases risk)
# colour: red = high, blue = low
class_names = target_encoder.classes_  

for i, class_name in enumerate(class_names):
    plt.figure(figsize = (10, 8))
    shap.summary_plot(shap_values_clf[i], X_test_sample, plot_type = "dot", max_display = 15, show = False)
    plt.title(f"SHAP Beeswarm Plot for Class: {class_name}", fontsize = 15)
    plt.tight_layout()
    plt.savefig(f"shap_beeswarm_{class_name}.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved SHAP beeswarm plot for class '{class_name}' as 'shap_beeswarm_{class_name}.png'") 

In [ ]:
# waterfall plot

patient_index = 0

# shap.explanation wraps values, base values and data into one object
clf_explaination = shap.Explanation(values = shap_values_clf[:, patient_index, :] ,base_values = clf_explainer.expected_value, data = X_test_sample.iloc[patient_index].values, feature_names = FEATURES)

# the waterfall plot
plt.figure(figsize = (10, 6))
shap.waterfall_plot(clf_explaination[0], show = False)
plt.title(f"SHAP Waterfall - Patient {patient_index}, Predicted class: {class_names[0]}", fontsize = 12)
plt.tight_layout()
plt.savefig(f"shap_waterfall_patient_{patient_index}.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved SHAP waterfall plot for patient index {patient_index} as 'shap_waterfall_patient_{patient_index}.png'")

In [ ]:
# Print the top features table

mean_abs_shap = np.mean([np.abs(shap_values_clf[i]) for i in range(len(class_names))], axis = (0,1))

# means_abs_shap is now a 1D array of length n_features

features_importance_df = pd.DataFrame({"Feature": FEATURES, "Mean SHAP Value": mean_abs_shap}).sort_values("Mean SHAP Value", ascending = False)

# saving the table for the report
features_importance_df.to_csv("shap_feature_importance.csv", index = False)

In [ ]:
# SHAP for Patient Segmentation
# we use KernelExplainer which is a model-agnostic which means it works with anything

print("\n Computing SHAP values for Patient Segmentation (k-means)")

# k-means uses eculiden distance
# StandardScaler transforms each column to mean = 0 and std = 1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns = FEATURES)

# Training k-means with k = 3
kmeans = KMeans(n_clusters = 3, n_init = 10, random_state = 42)
kmeans.fit(X_scaled_df)

# Attaching cluster labels to the original data
df["cluster"] = kmeans.labels_
print("K-means cluster distribution: ")
print(pd.Series(kmeans.labels_).value_counts().sort_index())

In [ ]:
# Define a prediction function for KernelEcplainer
# KernelExplainer needs a FUNCTION that takes input X and returns predictions.
# For K-Means we return the cluster assignment as a number.
def kmeans_predict(X_input):
    return kmeans.predict(X_input).astype(float)

In [ ]:
# Creating Background data (summary)
# the KernalExplainer needs a background dataset to compute SHAP values. This is typically a summary of the training data.

background_data = shap.kmeans(X_scaled_df, 50) # the 50 represents the background samples

In [ ]:
# Create the KernelExplainer
seg_explainer = shap.KernelExplainer(kmeans_predict, background_data)

In [ ]:
# Compute SHAP values on a small sample
X_seg_sample = X_scaled_df.sample(n = 100, random_state = 42)
shap_values_seg = seg_explainer.shap_values(X_seg_sample, nsamples = 100)
print(f"Segmentation SHAP values shape: {np.array(shap_values_seg).shape}")

In [ ]:
# Segmentation Global Feature Importance
# This shows which features matter most in SEPARATING the 3 patient clusters.
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values_seg, X_seg_sample, plot_type = "bar", max_display = 15, show = False)
plt.title("Global Feature Importance — Patient Lifestyle Segmentation (K-Means)", fontsize = 13)
plt.tight_layout()
plt.savefig("outputs/shap_segmentation_global_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Segmentation Beeswarm Plot
# shows direction: does high BMI push a patient toward a higher cluster number
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values_seg, X_seg_sample, plot_type = "dot",max_display = 15, show = False)
plt.title("SHAP Beeswarm — Patient Lifestyle Segmentation", fontsize=13)
plt.tight_layout()
plt.savefig("outputs/shap_segmentation_beeswarm.png", dpi=150, bbox_inches="tight")
plt.show()

# printing top segmentation features
seg_importance_df = pd.DataFrame({"Feature": FEATURES, "Mean |SHAP|": np.mean(np.abs(shap_values_seg), axis = 0)}).sort_values("Mean |SHAP|", ascending = False)
 
print("\n Top 10 Features Driving Cluster Separation:")
print(seg_importance_df.head(10).to_string(index = False))
 
seg_importance_df.to_csv("outputs/shap_segmentation_feature_importance.csv", index = False)

In [ ]:
# Combined summary - Report Ready
#Merge both impance tables died by side for easy comparision in the report
combined = feature_importance_df.rename(columns = {"Mean SHAP": "Classification SHAP"}).merge(
    seg_importance_df.rename(columns={"Mean SHAP": "Segmentation SHAP"}),
    on="Feature"
).sort_values("Classification SHAP", ascending = False)
 
print("\n Combined Feature Importance (Top 10):")
print(combined.head(10).to_string(index=False))
 
combined.to_csv("outputs/shap_combined_importance.csv", index = False)
print(" Saved: shap_combined_importance.csv")
 
print("\n SHAP Analysis Complete. All outputs saved to outputs/ folder.")